The idea is to first plan a route && create the speed profile according to the lane then:
- Check for stop signs that apply
- If they apply -> truncate the path at the first stop sign
- Set a speed profile element at the final waypoint with speed = 0

The planner will need to keep track of it's state. From P&C Design:
- There are 3 states: Driving, Slowing, Stopped
- Only the following transitions are allowed:
    - Driving --> Slowing
    - Slowing --> Stopped
    - Stopped --> Driving

Important changes:
- Speed profile and planned path are consolidated into a single data structure `route`.
    - Route is a list of waypoints. Each waypoint has a position and target speed.

In [5]:
from dataclasses import dataclass
from enum import Enum
from time import time, sleep
from math import sqrt, pow

class State(Enum):
    DRIVING = 'driving'
    STOPPING  = 'stopping'
    STOPPED = 'stopped'

@dataclass
class Waypoint:
    position: tuple[float, float]
    speed: float

class Control:
    def __init__(self):
        self.route = []
        self.speed = 0

    def set_route(self, route: list[Waypoint]):
        print('Received route from planning.')
        self.route = route

    def brake(self, amt):
        pass

class Planner:
    def __init__(self, control: Control):
        self._state = State.STOPPED
        self._stop_time = 0
        self.STOP_WAIT_TIME = 1 # wait 1 second
        self.control = control
        self._target_speed = 1

    def start(self):
        num_iterations = 20
        wait_time = 0.5 # seconds

        print(f'Running for a total {num_iterations * wait_time} seconds')

        i = 0
        while i < num_iterations:
            self._run()
            sleep(0.5)
            i += 1


    def _run(self):
        """Run the planning loop. This should be run pretty infrequently since dead-reckoning in control should handle small movements."""
        # the behaviour depends on the state
        # also define state transitions
        match self._state:
            case State.DRIVING:
                return self._drive()
            case State.STOPPING:
                return self._stopping()
            case State.STOPPED:
                return self._stopped()
            case _:
                print('ERROR: unknown state!')

    def _drive(self):
        # state transitions
        sign = self._get_next_applicable_sign()
        if sign:
            # there's a stop sign coming up!
            # transition to stopping
            print('Transitioning from DRIVING to STOPPING')
            self._state = State.STOPPING
            return self._run()

        # plan our route
        route = self._plan_route()
        self._send_to_control(route)

    def _stopping(self):
        # state transitions
        if self.control.speed == 0:
            print('Transitioning from STOPPING to STOPPED')
            
            # pin stop time
            self._stop_time = time()

            # transition
            self._state = State.STOPPED 
            return self._run()

        # plan our route
        route: list[Waypoint] = self._plan_route()

        # truncate route at stop sign
        stop_waypoint_idx = self._determine_stop_waypoint(route, self._get_next_applicable_sign())

        route = route[:stop_waypoint_idx]
        route[-1].speed = 0
        route = self._balance_route(route) # since our speed profile abruptly ends in a 0 now it should be rebalanced to smooth this out

    def _stopped(self):
        if self.control.speed > 0:
            print('WARN: Car is moving while stopped!')
            self.control.brake(1.0)

        # check the timer, if it's done then transition to driving
        if time() - self._stop_time > self.STOP_WAIT_TIME:
            print('Transitioning from STOPPED to DRIVING')
            self._state = State.DRIVING
            return self._run

    def _determine_stop_waypoint(self, route: list[Waypoint], sign):
        def find_closest_waypoint(route: list[Waypoint], sign):
            min_dist = None
            min_waypoint_idx = -1
            for idx, waypoint in enumerate(route):
                dist = sqrt(
                    pow(waypoint.position[0] - sign[0], 2),
                    pow(waypoint.position[1] - sign[1], 2),
                )

                if min_dist is None or dist < min_dist:
                    min_dist = dist 
                    min_waypoint_idx = idx
                    continue 

            return min_waypoint_idx
        
        min_waypoint_idx = find_closest_waypoint(route, sign)
        return min_waypoint_idx

    def _balance_route(self):
        pass

    def _get_next_applicable_sign(self):
        return (1, 1)

    def _plan_route(self) -> list[Waypoint]:
        return [Waypoint((0, i * 1), self._target_speed) for i in range(1, 11)]

    def _send_to_control(self, route):
        self.control.set_route(route)



In [6]:
control = Control()
planner = Planner(control)
planner.start()

Running for a total 10.0 seconds
Transitioning from STOPPED to DRIVING
Transitioning from DRIVING to STOPPING
Transitioning from STOPPING to STOPPED
Transitioning from STOPPED to DRIVING
Transitioning from DRIVING to STOPPING
Transitioning from STOPPING to STOPPED
Transitioning from STOPPED to DRIVING
Transitioning from DRIVING to STOPPING
Transitioning from STOPPING to STOPPED
Transitioning from STOPPED to DRIVING
Transitioning from DRIVING to STOPPING
Transitioning from STOPPING to STOPPED
Transitioning from STOPPED to DRIVING
Transitioning from DRIVING to STOPPING
Transitioning from STOPPING to STOPPED
Transitioning from STOPPED to DRIVING
Transitioning from DRIVING to STOPPING
Transitioning from STOPPING to STOPPED
Transitioning from STOPPED to DRIVING
Transitioning from DRIVING to STOPPING
Transitioning from STOPPING to STOPPED


Improvements:
- We could create a state system class that manages and logs all of these states and their transitions. But I'm not gonna do that bc it's a lot of extra work and not really necessary.